In [ ]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [ ]:
df["sentiment"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
print(df.head())

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=42
)

In [ ]:
import re

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b[a-zA-Z0-9]+\b", text)
    return [token for token in tokens if len(token) > 2]

print(tokenize(data["review"].iloc[0]))

['one', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'episode', 'you', 'hooked', 'they', 'are', 'right', 'this', 'exactly', 'what', 'happened', 'with', 'the', 'first', 'thing', 'that', 'struck', 'about', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'violence', 'which', 'set', 'right', 'from', 'the', 'word', 'trust', 'this', 'not', 'show', 'for', 'the', 'faint', 'hearted', 'timid', 'this', 'show', 'pulls', 'punches', 'with', 'regards', 'drugs', 'sex', 'violence', 'its', 'hardcore', 'the', 'classic', 'use', 'the', 'word', 'called', 'that', 'the', 'nickname', 'given', 'the', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'focuses', 'mainly', 'emerald', 'city', 'experimental', 'section', 'the', 'prison', 'where', 'all', 'the', 'cells', 'have', 'glass', 'fronts', 'and', 'face', 'inwards', 'privacy', 'not', 'high', 'the', 'agenda', 'city', 'home', 'many', 'aryans', 'muslims', 'gangstas', 'latinos', 'christians', 'italians', 'ir

In [ ]:
from collections import Counter

word_counter = Counter()

for review in X_train:
    word_counter.update(tokenize(review))

VOCAB_SIZE = 10000

common_words = word_counter.most_common(VOCAB_SIZE - 2)

word_to_idx = {
    word: i + 2 for i, (word, _) in enumerate(common_words)
}

# Special tokens
word_to_idx.update({
    "<PAD>": 0,
    "<OOV>": 1
})


In [ ]:
print(list(word_to_idx)[:10])

['the', 'and', 'this', 'that', 'was', 'movie', 'for', 'with', 'but', 'film']


In [ ]:
MAX_LEN = 256

def encode(text):
    return [
        word_to_idx.get(word, word_to_idx["<OOV>"])
        for word in tokenize(text)
    ]

def pad_sequence(seq, max_len=MAX_LEN):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))

X_train_seq = [pad_sequence(encode(text)) for text in X_train]
X_test_seq = [pad_sequence(encode(text)) for text in X_test]

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.long)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_seq, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

# Quick check
for xb, yb in train_loader:
    print(xb.shape)
    break

In [ ]:
import torch.nn as nn

class BaseModel(nn.Module):
    def __init__(self, rnn_type, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        if rnn_type == "RNN":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == "LSTM":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == "GRU":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return torch.sigmoid(out)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate(model, loader, device):
    model.eval()

    preds_list = []
    labels_list = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.unsqueeze(1).to(device)

            outputs = model(xb)
            preds = (outputs > 0.5).float()

            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(yb.cpu().numpy())

    return {
        "accuracy": accuracy_score(labels_list, preds_list),
        "precision": precision_score(labels_list, preds_list, zero_division=0),
        "recall": recall_score(labels_list, preds_list, zero_division=0),
        "f1": f1_score(labels_list, preds_list, zero_division=0),
        "cm": confusion_matrix(labels_list, preds_list)
    }

In [ ]:
def train(model, train_loader, test_loader, device, lr=1e-3, epochs=10):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    for ep in range(epochs):
        model.train()
        total_loss = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.unsqueeze(1).to(device)

            optimizer.zero_grad()

            preds = model(xb)
            loss = loss_fn(preds, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {ep+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    return evaluate(model, test_loader, device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

models = {
    "RNN": BaseModel("RNN", VOCAB_SIZE, 100, 128),
    "LSTM": BaseModel("LSTM", VOCAB_SIZE, 100, 128),
    "GRU": BaseModel("GRU", VOCAB_SIZE, 100, 128),
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    results[name] = train(model, train_loader, test_loader, device, epochs=20)

for name, res in results.items():
    print(f"\n{name} Results")
    print("Accuracy:", res["accuracy"])
    print("Precision:", res["precision"])
    print("Recall:", res["recall"])
    print("F1:", res["f1"])
    print("Confusion Matrix:\n", res["cm"])

Using device: cuda


In [ ]:
for name, model in models.items():
    print(f"\nTraining {name} on {device}")
    acc, precision, recall, f1, cm = train_model(model, train_loader, test_loader, epochs=epochs)
    results[name] = (acc, precision, recall, f1, cm)


Training RNN on cuda
Epoch 1/20 - Loss: 0.6949
Epoch 2/20 - Loss: 0.6946
Epoch 3/20 - Loss: 0.6939
Epoch 4/20 - Loss: 0.6934
Epoch 5/20 - Loss: 0.6900
Epoch 6/20 - Loss: 0.6830
Epoch 7/20 - Loss: 0.6778
Epoch 8/20 - Loss: 0.6705
Epoch 9/20 - Loss: 0.6579
Epoch 10/20 - Loss: 0.6481
Epoch 11/20 - Loss: 0.6422
Epoch 12/20 - Loss: 0.6385
Epoch 13/20 - Loss: 0.6281
Epoch 14/20 - Loss: 0.6181
Epoch 15/20 - Loss: 0.6138
Epoch 16/20 - Loss: 0.6036
Epoch 17/20 - Loss: 0.5951
Epoch 18/20 - Loss: 0.5878
Epoch 19/20 - Loss: 0.5813
Epoch 20/20 - Loss: 0.5774

Training LSTM on cuda
Epoch 1/20 - Loss: 0.6903
Epoch 2/20 - Loss: 0.6721
Epoch 3/20 - Loss: 0.6463
Epoch 4/20 - Loss: 0.6405
Epoch 5/20 - Loss: 0.5669
Epoch 6/20 - Loss: 0.5028
Epoch 7/20 - Loss: 0.3334
Epoch 8/20 - Loss: 0.2593
Epoch 9/20 - Loss: 0.2236
Epoch 10/20 - Loss: 0.1849
Epoch 11/20 - Loss: 0.1469
Epoch 12/20 - Loss: 0.1120
Epoch 13/20 - Loss: 0.0840
Epoch 14/20 - Loss: 0.0692
Epoch 15/20 - Loss: 0.0577
Epoch 16/20 - Loss: 0.0473
E

In [ ]:
for name, (acc, precision, recall, f1, cm) in results.items():

    print(f"\n{name} Results")
    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)
    print("Confusion Matrix:\n", cm)


RNN Results
Accuracy: 0.4919
Precision: 0.49380896226415094
Recall: 0.3324072236554872
F1-score: 0.39734313841774405
Confusion Matrix:
 [[3244 1717]
 [3364 1675]]

LSTM Results
Accuracy: 0.8647
Precision: 0.874289195775792
Recall: 0.8543361778130582
F1-score: 0.8641975308641975
Confusion Matrix:
 [[4342  619]
 [ 734 4305]]

GRU Results
Accuracy: 0.8791
Precision: 0.8935470612412659
Recall: 0.8628696169874975
F1-score: 0.8779404341241797
Confusion Matrix:
 [[4443  518]
 [ 691 4348]]
